# Práctica 2. Representación algebraica y geométrica de un qubit

**Tema:** Conceptos fundamentales. Superposición cuántica  
**Duración estimada:** 60–75 minutos  
**Modalidad:** Individual  
**Lenguaje:** Python  
**Bibliotecas:** NumPy y Matplotlib  
**Uso de Qiskit:** No se utilizará en esta práctica.

En esta práctica trabajaremos con números complejos, notación de Dirac, productos internos, bases ortonormales, fase global y esfera de Bloch.

## Objetivos

Al finalizar la práctica serás capaz de:

- Calcular conjugados, módulos y fases de números complejos.
- Diferenciar $z^2$ y $|z|^2$.
- Representar *kets* y *bras* con NumPy.
- Calcular productos internos y comprobar la ortogonalidad.
- Validar bases ortonormales.
- Distinguir fase global y fase relativa.

> Completa las celdas marcadas con `TODO` y responde en Markdown a todas las preguntas.

## Entrega

Entrega este notebook con el nombre:

```text
practica_02_representacion_qubit_apellidos_nombre.ipynb
```

Debe contener código ejecutable, resultados, gráficas, respuestas razonadas, pruebas y una conclusión final. No se considerará completa una entrega formada únicamente por código.

## Preparación del entorno

In [3]:
pip install matplotlib

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------  9.4/9.5 MB 49.2 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 44.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 40.5 MB/s  0:00:00
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------- ----------- 5.2/7.2 MB 41.8 MB/s eta 0:00:01
   ------------------------------ --------- 5.5/7.2 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 10.5 MB/s  0:00:00

   ---------------------------------------- 0/7 [pyparsing]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pi

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)
TOLERANCIA = 1e-10


def es_estado_valido(estado, tolerancia=1e-10):
    try:
        vector = np.asarray(estado, dtype=complex)
    except (TypeError, ValueError):
        return False

    if vector.shape != (2,) or not np.all(np.isfinite(vector)):
        return False

    norma = np.linalg.norm(vector)
    return (
        not np.isclose(norma, 0.0, atol=tolerancia)
        and np.isclose(norma, 1.0, atol=tolerancia, rtol=0.0)
    )


def normalizar_estado(estado):
    vector = np.asarray(estado, dtype=complex)

    if vector.shape != (2,):
        raise ValueError("Un qubit debe tener dos amplitudes")
    if not np.all(np.isfinite(vector)):
        raise ValueError("Las amplitudes deben ser finitas")

    norma = np.linalg.norm(vector)
    if np.isclose(norma, 0.0):
        raise ValueError("El vector nulo no puede normalizarse")

    return vector / norma


ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# Parte A. Números complejos

## Ejercicio 1. Componentes de un número complejo

Considera:

$$z_1=3+4i,\qquad z_2=1-i,\qquad z_3=-2i,\qquad z_4=-1-\sqrt{3}i.$$

Para cada número muestra su parte real, parte imaginaria, conjugado, módulo, módulo al cuadrado, fase en radianes y fase en grados.

In [20]:
z_1 = np.array(3 + 4j, dtype=complex)
z_2 = np.array(1 - 1j, dtype=complex)
z_3 = np.array(-2j, dtype=complex)
z_4 = np.array(-1 - np.sqrt(3) * 1j, dtype=complex)

numeros = {"z_1": z_1, "z_2": z_2, "z_3": z_3, "z_4": z_4}

for nombre, z in numeros.items():
    print(f"{nombre}: {z}")
    print(f"Parte real: {z.real}")
    print(f"Parte imaginaria: {z.imag}")
    print(f"Conjugado: {z.conjugate()}")
    print(f"Módulo: {np.abs(z)}")
    print(f"Módulo al cuadrado: {np.abs(z)**2}")
    print(f"Fase(rad): {np.angle(z)}")
    print(f"Fase(deg): {np.degrees(np.angle(z))}")
    print()

z_1: (3+4j)
Parte real: 3.0
Parte imaginaria: 4.0
Conjugado: (3-4j)
Módulo: 5.0
Módulo al cuadrado: 25.0
Fase(rad): 0.9272952180016122
Fase(deg): 53.13010235415598

z_2: (1-1j)
Parte real: 1.0
Parte imaginaria: -1.0
Conjugado: (1+1j)
Módulo: 1.4142135623730951
Módulo al cuadrado: 2.0000000000000004
Fase(rad): -0.7853981633974483
Fase(deg): -45.0

z_3: (-0-2j)
Parte real: -0.0
Parte imaginaria: -2.0
Conjugado: (-0+2j)
Módulo: 2.0
Módulo al cuadrado: 4.0
Fase(rad): -1.5707963267948966
Fase(deg): -90.0

z_4: (-1-1.7320508075688772j)
Parte real: -1.0
Parte imaginaria: -1.7320508075688772
Conjugado: (-1+1.7320508075688772j)
Módulo: 2.0
Módulo al cuadrado: 4.0
Fase(rad): -2.0943951023931957
Fase(deg): -120.00000000000001



**Responde:**

1. ¿Puede el módulo ser negativo?
    - No.
2. ¿Qué relación existe entre la fase de un número y la de su conjugado?
    - Son opuestas. El conjugado tiene un cambio de signo en su parte real, por ello se invierte.
3. ¿En qué cuadrante se encuentra cada número?
    - z1: en el primer cuadrante.
    - z2: en el cuarto cuadrante.
    - z3: en el eje imaginario negativo.
    - z4: en el tercer cuadrante.
4. ¿Por qué conviene utilizar `np.angle()` en lugar de calcular simplemente $\arctan(b/a)$?
    - Para que se devuelvan los angulos entre $-\pi$ y $\pi$. $\arctan(b/a)$ los devuelve entre $-\pi/2$ y $\pi/2$

## Ejercicio 2. Potencias de $i$

Calcula manualmente $i^{15}$, $i^{26}$, $i^{51}$ e $i^{100}$. Después comprueba los resultados con Python.

In [22]:
potencias = [15, 26, 51, 100]

# TODO: calcula y muestra 1j ** exponente para cada valor
for exp in potencias:
    resultado = 1j ** exp
    print(f"1j ^ {exp} = {resultado}")

1j ^ 15 = (-0-1j)
1j ^ 26 = (-1+0j)
1j ^ 51 = (-0-1j)
1j ^ 100 = (1+0j)


**Responde:** ¿Cada cuántas potencias se repite el patrón? ¿Cómo interviene el resto de dividir el exponente entre 4?

    - Cada 4 potencias. El resto de dividir el exponente entre 4 sirve para saber el cuadrante en el que estamos.

## Ejercicio 3. Cuadrado y módulo al cuadrado

Para $z=(1+i)/2$, calcula manualmente $z^2$, $z^*$, $z^*z$, $|z|$ y $|z|^2$. Después compruébalo con NumPy.

In [23]:
z = (1 + 1j) / 2

cuadrado = z ** 2
conjugado = np.conj(z)
producto_conjugado = z * conjugado
modulo = np.abs(z)
modulo_cuadrado = np.abs(z) ** 2

print("z²:", cuadrado)
print("z*:", conjugado)
print("z*z:", producto_conjugado)
print("|z|:", modulo)
print("|z|²:", modulo_cuadrado)

z²: 0.5j
z*: (0.5-0.5j)
z*z: (0.5+0j)
|z|: 0.7071067811865476
|z|²: 0.5000000000000001


**Responde:** ¿Por qué $z^2$ no puede utilizarse como probabilidad? ¿Por qué $z^*z$ siempre es real y no negativo? ¿Puede una amplitud tener módulo mayor que 1 en un estado normalizado?

**Respuesta:** $z^2$ puede ser negativo o imaginario, por ser complejo. Necesitamos que la probabilidad esté entre 0 y 1. $z*z$ es una suma de cuadrados reales: $a^2+b^2$. En un estado normalizado como máximo, el módulo será 1.

## Ejercicio 4. Forma cartesiana y polar

Implementa funciones para obtener el módulo y la fase de un número complejo y para reconstruirlo a partir de su forma polar.

In [34]:
def datos_polares(z):
    """Devuelve módulo, fase en radianes y fase en grados."""
    z = complex(z)
    modulo = (float)(np.abs(z))
    fase_rad = (float)(np.angle(z))
    fase_grad = np.degrees(fase_rad)
    return modulo, fase_rad, fase_grad

def crear_complejo(modulo, fase):
    """Construye un complejo a partir de módulo y fase."""
    return complex(modulo * np.exp(1j * fase))


numeros_polares = [1 + 1j, -1 + 1j, -1j, 2 * np.exp(1j * np.pi / 3)]

for z in numeros_polares:
    modulo, fase_rad, fase_grad = datos_polares(z)
    print(f"z: {z:<10f}, módulo: {modulo:<10f}, fase(rad): {fase_rad:<10f}, fase(deg): {fase_grad:<10f}")
    z_reconstruido = crear_complejo(modulo, fase_rad)
    print(f"Reconstruido: {z_reconstruido:<10f}")
    print()

z: 1.000000+1.000000j, módulo: 1.414214  , fase(rad): 0.785398  , fase(deg): 45.000000 
Reconstruido: 1.000000+1.000000j

z: -1.000000+1.000000j, módulo: 1.414214  , fase(rad): 2.356194  , fase(deg): 135.000000
Reconstruido: -1.000000+1.000000j

z: -0.000000-1.000000j, módulo: 1.000000  , fase(rad): -1.570796 , fase(deg): -90.000000
Reconstruido: 0.000000-1.000000j

z: 1.000000+1.732051j, módulo: 2.000000  , fase(rad): 1.047198  , fase(deg): 60.000000 
Reconstruido: 1.000000+1.732051j



**Responde:** ¿Por qué dos fases que difieren en $2\pi$ representan el mismo número complejo?

**Respuesta:** Porque $e^{i\theta} = \cos{\theta} + i\sin{\theta}$ es periódica de periodo $2\pi:\theta$ y $\theta+2\pi$ dan el mismo punto del plano complejo.

# Parte B. Notación de Dirac y producto interno

## Ejercicio 5. Del *ket* al *bra*

Representa

$$|\psi\rangle=\frac{1}{\sqrt2}\begin{pmatrix}1\\i\end{pmatrix}$$

como array unidimensional y como matriz columna. Obtén el *bra* asociado en ambos casos y muestra sus formas con `shape`.

In [38]:
# TODO: crea ambas representaciones
psi_vector = np.array([1, 1j], dtype=complex) / np.sqrt(2)
psi_columna = psi_vector.reshape(2, 1)

# TODO: calcula los bras correspondientes
bra_vector = np.conjugate(psi_vector)
bra_matriz = np.conjugate(psi_columna).T

# TODO: muestra valores y shapes
print(f"Ket vector: {psi_vector}, shape: {psi_vector.shape}")
print(f"Ket columna:\n{psi_columna}, shape: {psi_columna.shape}")
print(f"Bra vector: {bra_vector}, shape: {bra_vector.shape}")
print(f"Bra matriz:\n{bra_matriz}, shape: {bra_matriz.shape}")

Ket vector: [0.70711+0.j      0.     +0.70711j], shape: (2,)
Ket columna:
[[0.70711+0.j     ]
 [0.     +0.70711j]], shape: (2, 1)
Bra vector: [0.70711-0.j      0.     -0.70711j], shape: (2,)
Bra matriz:
[[0.70711-0.j      0.     -0.70711j]], shape: (1, 2)


**Responde:**

1. ¿Por qué `psi_vector.T` no cambia la forma del array?
    - Porque es un array de 1 sola dimension (2,), no hay  filas y columnas que intercambiar
2. ¿Qué operación debe realizarse además de la transposición?
    - Hay que conjugarlo
3. ¿Qué formas tienen el *ket* columna y su *bra*?
    - El ket tiene la forma (2, 1) y el bra fila tiene forma (1, 2)

## Ejercicio 6. Producto interno

Implementa $\langle a|b\rangle$ sin utilizar inicialmente `np.vdot()`. Recuerda conjugar el primer vector.

In [41]:
def producto_interno(estado_a, estado_b):
    """Calcula <estado_a|estado_b>."""
    a = np.asarray(estado_a, dtype=complex)
    b = np.asarray(estado_b, dtype=complex)
    if a.shape != b.shape or a.ndim != 1 or b.ndim != 1:
        raise ValueError("Los estados deben tener la misma dimensión")
    return complex(np.sum(np.conj(a) * b))

kets = {"0": ket_0, "1": ket_1}
for (nombre_a, estado_a) in kets.items():
    for nombre_b, estado_b in kets.items():
        resultado = producto_interno(estado_a, estado_b)
        resultado_vdot = np.vdot(estado_a, estado_b)
        print(f"<{nombre_a}|{nombre_b}> = {resultado}, coincide con np.vdot(): {np.isclose(resultado, resultado_vdot)}")

<0|0> = (1+0j), coincide con np.vdot(): True
<0|1> = 0j, coincide con np.vdot(): True
<1|0> = 0j, coincide con np.vdot(): True
<1|1> = (1+0j), coincide con np.vdot(): True


## Ejercicio 7. Propiedades del producto interno

Construye

$$|\psi\rangle=\frac{|0\rangle+i|1\rangle}{\sqrt2},\qquad
|\phi\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}.$$

Comprueba que $\langle\psi|\phi\rangle=\langle\phi|\psi\rangle^*$.

In [43]:
psi = (ket_0 + 1j * ket_1) / np.sqrt(2)
phi = (ket_0 - ket_1) / np.sqrt(2)

producto_psi_phi = producto_interno(psi, phi)
producto_phi_psi = producto_interno(phi, psi)

print("<psi|phi> =", producto_psi_phi)
print("<phi|psi> =", producto_phi_psi)

assert np.isclose(producto_psi_phi, np.conj(producto_phi_psi))

<psi|phi> = (0.4999999999999999+0.4999999999999999j)
<phi|psi> = (0.4999999999999999-0.4999999999999999j)


**Responde:** ¿El producto interno debe ser real? ¿Qué propiedad se cumple al intercambiar los estados? ¿Qué error aparece si no se conjuga el primer vector?

**Respuesta:** 
- No. En general es un complejo. Solo es real cuando se calcula $\langle\psi|\psi\rangle=\lVert\psi\rVert^2$. 
- Se cumple la simetría conjugada. 
- No da una norma real y no negativa (para |+i) sale 0.

# Parte C. Ortogonalidad y bases

## Ejercicio 8. Bases $X$ e $Y$

Construye los estados $|+\rangle$, $|-\rangle$, $|+i\rangle$ y $|-i\rangle$. Para cada uno muestra el vector, la norma, las probabilidades en la base computacional y su fase relativa.

In [ ]:
ket_plus = (ket_0 + ket_1) / np.sqrt(2)
ket_minus = (ket_0 - ket_1) / np.sqrt(2)
ket_mas_i = (ket_0 + 1j * ket_1) / np.sqrt(2)
ket_menos_i = (ket_0 - 1j * ket_1) / np.sqrt(2)

estados_bases = {
    "|+>": ket_plus,
    "|->": ket_minus,
    "|+i>": ket_mas_i,
    "|-i>": ket_menos_i,
}

for nombre, estado in estados_bases.items():
    p0, p1 = np.abs(estado) ** 2
    fase_rel = np.degrees(np.angle(np.conj(estado[0]) * estado[1]))
    print(f"Estado: {nombre}: vector: {estado}, norma: {np.linalg.norm(estado):.5f}")
    print(f"Probabilidades: |0>: {p0:.5f}, |1>: {p1:.5f}, fase relativa: {fase_rel:.5f}°")
    print()

Estado: |+>: vector: [0.70711+0.j 0.70711+0.j], norma: 1.00000
Probabilidades: |0>: 0.50000, |1>: 0.50000, fase relativa: 0.00000°

Estado: |->: vector: [ 0.70711+0.j -0.70711+0.j], norma: 1.00000
Probabilidades: |0>: 0.50000, |1>: 0.50000, fase relativa: 180.00000°

Estado: |+i>: vector: [0.70711+0.j      0.     +0.70711j], norma: 1.00000
Probabilidades: |0>: 0.50000, |1>: 0.50000, fase relativa: 90.00000°

Estado: |-i>: vector: [0.70711+0.j      0.     -0.70711j], norma: 1.00000
Probabilidades: |0>: 0.50000, |1>: 0.50000, fase relativa: -90.00000°



**Responde:** ¿Producen los cuatro estados las mismas probabilidades en la base computacional? ¿Son el mismo estado? ¿Qué información los diferencia? ¿Qué pares deben ser ortogonales?

**Respuesta:** 
- Los 4 dan P(0) = P(1) = 1/2
- No. Son distintos.
- La fase relativa entre amplitudes.
- |+> con |-> y |+i> con |-i>

## Ejercicio 9. Comprobación de ortogonalidad

Implementa una función que valide los estados y determine si su producto interno es aproximadamente cero.

In [50]:
def son_ortogonales(estado_a, estado_b, tolerancia=1e-10):
    """Devuelve True si dos estados válidos son ortogonales."""
    if not (es_estado_valido(estado_a) and es_estado_valido(estado_b)):
        raise ValueError("Ambos estados deben ser válidos y normalizados")
    return bool(abs(producto_interno(estado_a, estado_b)) <= tolerancia)


pares = [
    ("|0>, |1>", ket_0, ket_1),
    ("|+>, |->", ket_plus, ket_minus),
    ("|+i>, |-i>", ket_mas_i, ket_menos_i),
    ("|0>, |+>", ket_0, ket_plus),
    ("|1>, |->", ket_1, ket_minus),
    ("|+>, |+i>", ket_plus, ket_mas_i),
]

for nombre, a, b in pares:
    p = producto_interno(a, b)
    print(f"{nombre:12s}, <a|b> = {p:.5f}, |<a|b>| = {np.abs(p):.5f}, ortogonales: {son_ortogonales(a, b)}")

|0>, |1>    , <a|b> = 0.00000+0.00000j, |<a|b>| = 0.00000, ortogonales: True
|+>, |->    , <a|b> = 0.00000+0.00000j, |<a|b>| = 0.00000, ortogonales: True
|+i>, |-i>  , <a|b> = 0.00000+0.00000j, |<a|b>| = 0.00000, ortogonales: True
|0>, |+>    , <a|b> = 0.70711+0.00000j, |<a|b>| = 0.70711, ortogonales: False
|1>, |->    , <a|b> = -0.70711+0.00000j, |<a|b>| = 0.70711, ortogonales: False
|+>, |+i>   , <a|b> = 0.50000+0.50000j, |<a|b>| = 0.70711, ortogonales: False


## Ejercicio 10. Base ortonormal

Implementa una función que compruebe que dos estados están normalizados y son ortogonales.

In [51]:
def es_base_ortonormal(estado_a, estado_b, tolerancia=1e-10):
    """Determina si dos estados forman una base ortonormal de un qubit."""
    if not (es_estado_valido(estado_a, tolerancia) and es_estado_valido(estado_b, tolerancia)):
        return False
    return son_ortogonales(estado_a, estado_b, tolerancia)


bases = {
    "base_Z": (ket_0, ket_1),
    "base_X": (ket_plus, ket_minus),
    "base_Y": (ket_mas_i, ket_menos_i),
    "base_repetida": (ket_0, ket_0),
    "no_ortogonal": (ket_0, ket_plus),
    "no_normalizada": (np.array([1, 1], dtype=complex), ket_1),
}

for nombre, (a, b) in bases.items():
    print(f"{nombre:15s} -> {str(es_base_ortonormal(a, b)):5s}")

base_Z          -> True 
base_X          -> True 
base_Y          -> True 
base_repetida   -> False
no_ortogonal    -> False
no_normalizada  -> False


## Ejercicio 11. Cambio de base

Los coeficientes de $|\psi\rangle$ en una base ortonormal $\{|u\rangle,|v\rangle\}$ son $\langle u|\psi\rangle$ y $\langle v|\psi\rangle$.

In [53]:
def coordenadas_en_base(estado, base):
    """Calcula las coordenadas de un estado en una base ortonormal."""
    u, v = base
    if not es_base_ortonormal(u, v):
        raise ValueError("La base no es ortonormal")
    if not es_estado_valido(estado):
        raise ValueError("El estado no es válido")
    return np.array([producto_interno(u, estado), producto_interno(v, estado)], dtype=complex)

base_Z = [ket_0, ket_1]
base_X = [ket_plus, ket_minus]
base_Y = [ket_mas_i, ket_menos_i]

casos = [
    ("|0> en base X", ket_0, base_X),
    ("|1> en base X", ket_1, base_X),
    ("|+> en base Z", ket_plus, base_Z),
    ("|-i> en base Y", ket_menos_i, base_Y),
]

for nombre, estado, base in casos:
    c = coordenadas_en_base(estado, base)
    reconstruido = c[0] * base[0] + c[1] * base[1]
    print(f"{nombre:10s} -> coordenadas: {c}, reconstruido correcta={np.allclose(reconstruido, estado)}")

|0> en base X -> coordenadas: [0.70711+0.j 0.70711+0.j], reconstruido correcta=True
|1> en base X -> coordenadas: [ 0.70711+0.j -0.70711+0.j], reconstruido correcta=True
|+> en base Z -> coordenadas: [0.70711+0.j 0.70711+0.j], reconstruido correcta=True
|-i> en base Y -> coordenadas: [0.+0.j 1.+0.j], reconstruido correcta=True


**Responde:** ¿Cambiar de base modifica el estado físico? ¿Qué cambia? ¿Qué coordenadas debe tener $|-i\rangle$ en la base $Y$?

**Respuesta:**
- No. El estado físico es el mismo vector. Cambia la descripción.
- Cambian las coordenadas (los coeficientes), y las probabilidades de medir en esa base.
- (0, 1)

# Parte D. Fase global y fase relativa

## Ejercicio 12. Comparación de estados

Considera:

$$|\psi_A\rangle=\frac{1}{\sqrt2}(1,i)^T,\quad
|\psi_B\rangle=\frac{1}{\sqrt2}(i,-1)^T,$$

$$|\psi_C\rangle=\frac{1}{\sqrt2}(1,-i)^T,\quad
|\psi_D\rangle=-\frac{1}{\sqrt2}(1,i)^T.$$

Comprueba sus normas y probabilidades. Calcula el producto interno de $A$ con cada estado y determina cuáles son físicamente equivalentes a $A$.

In [60]:
psi_A = np.array([1, 1j], dtype=complex) / np.sqrt(2)
psi_B = np.array([1j, -1], dtype=complex) / np.sqrt(2)
psi_C = np.array([1, -1j], dtype=complex) / np.sqrt(2)
psi_D = -np.array([1, 1j], dtype=complex) / np.sqrt(2)

for nombre, e in {"psi_A": psi_A, "psi_B": psi_B, "psi_C": psi_C, "psi_D": psi_D}.items():
    p = producto_interno(psi_A, e)
    print(f"{nombre}: norma={np.linalg.norm(e):.5f}  P(0)={abs(e[0])**2:.5f}  P(1)={abs(e[1])**2:.5f}")
    print(f"   <A|{nombre[-1]}> = {p:.5f}   |<A|{nombre[-1]}>| = {abs(p):.5f}")
    print(f"equivalente a A: {np.isclose(abs(p), 1.0)}")
    print()
          
print()
print("B / A =", psi_B / psi_A, " (factor global i)")
print("D / A =", psi_D / psi_A, " (factor global -1)")
print("C / A =", psi_C / psi_A, " (no constante: fase relativa distinta)")

psi_A: norma=1.00000  P(0)=0.50000  P(1)=0.50000
   <A|A> = 1.00000+0.00000j   |<A|A>| = 1.00000
equivalente a A: True

psi_B: norma=1.00000  P(0)=0.50000  P(1)=0.50000
   <A|B> = 0.00000+1.00000j   |<A|B>| = 1.00000
equivalente a A: True

psi_C: norma=1.00000  P(0)=0.50000  P(1)=0.50000
   <A|C> = 0.00000+0.00000j   |<A|C>| = 0.00000
equivalente a A: False

psi_D: norma=1.00000  P(0)=0.50000  P(1)=0.50000
   <A|D> = -1.00000+0.00000j   |<A|D>| = 1.00000
equivalente a A: True


B / A = [0.+1.j 0.+1.j]  (factor global i)
D / A = [-1.+0.j -1.+0.j]  (factor global -1)
C / A = [ 1.+0.j -1.+0.j]  (no constante: fase relativa distinta)


**Responde:** ¿Qué factor relaciona $A$ con $B$? ¿Y $A$ con $D$? ¿Por qué $A$ y $C$ no son equivalentes aunque produzcan las mismas probabilidades?

**Respuesta:** TODO

## Ejercicio 13. Equivalencia salvo fase global

Para estados normalizados, utiliza el criterio $|\langle\psi|\phi\rangle|=1$. No dividas componentes, porque alguna podría ser cero.

In [61]:
def equivalentes_fase_global(estado_a, estado_b, tolerancia=1e-10):
    """Determina si dos estados válidos difieren solo en una fase global."""
    if not (es_estado_valido(estado_a, tolerancia) and es_estado_valido(estado_b, tolerancia)):
        raise ValueError("Ambos estados deben ser válidos y normalizados")
    producto = producto_interno(estado_a, estado_b)
    return bool(np.isclose(np.abs(producto), 1.0, atol=tolerancia, rtol=0.0))

fase = np.exp(1j * 0.7)

pruebas_equivalencia = [
    ("A con B (factor i)", psi_A, psi_A),
    ("A con D (factor -1)", psi_A, psi_D),
    ("A con A con fase =  e^(0.7i)", psi_A, fase * psi_A),
    ("|0> con -i|0> (componente nula)", ket_0, -1j * ket_0),
    ("A con C (fase relativa distinta)", psi_A, psi_C),
    ("|+> con |-> (ortogonales)", ket_plus, ket_minus),
    ("|0> con |+> (no ortogonales, no fase global)", ket_0, ket_plus)
]

for nombre, a, b in pruebas_equivalencia:
    resultado = equivalentes_fase_global(a, b)
    print(f"{nombre:40s} -> equivalentes: {resultado}")

A con B (factor i)                       -> equivalentes: True
A con D (factor -1)                      -> equivalentes: True
A con A con fase =  e^(0.7i)             -> equivalentes: True
|0> con -i|0> (componente nula)          -> equivalentes: True
A con C (fase relativa distinta)         -> equivalentes: False
|+> con |-> (ortogonales)                -> equivalentes: False
|0> con |+> (no ortogonales, no fase global) -> equivalentes: False


**Responde:** ¿Por qué es problemático dividir una componente entre otra? ¿Por qué el módulo del producto interno elimina la fase global? ¿Puede aplicarse directamente el criterio a vectores no normalizados?

**Respuesta:**
- Porque una componente puede ser 0, y con coma flotante se puede tener una cancelacion catastrófica.
- Porque la fase global es un factor $e^{i\theta}$: $\langle\psi|e^{i\theta}\psi\rangle=e^{i\theta}$ y su modulo es 1 para cualquier $\theta$
- No. Por la regla de Cauchy-Shwarz, $|\langle\psi|\phi\rangle|\le\lVert\psi\rVert\,\lVert\phi\rVert$. Hay que normalizar antes o comparar con $\lVert\psi\rVert\,\lVert\phi\rVert$